# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant -U

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets and their fields (`@id`).

To see which record sets and fields are available, we access `dataset.metadata.record_sets`, and for each, list its `@id` and the associated fields and their `@id`s.

In [ ]:
# List all record sets and fields by @id
print("Available Record Sets and Fields in Dataset:\n")

record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"Record Set: @id = {rs.id}, name = {rs.name}")
    if rs.fields:
        for f in rs.fields:
            print(f"  Field: @id = {f.id}, name = {f.name}, dataType = {f.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we automatically extract all record set `@id`s and create DataFrames for each, using the `@id` as the key. All fields are loaded for each record set.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record set @ids:", record_set_ids)

# Extract all available record sets into pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set: {rs_id} with columns: {list(dataframes[rs_id].columns)}")
    else:
        print(f"No records found for record set: {rs_id}")

# Choose one main record set for further analysis (using the first one if multiple)
main_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(main_record_set_id, pd.DataFrame())
if not df.empty:
    print(f"\nColumns in primary record set ({main_record_set_id}):")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No data available to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data for summary analysis.

We'll demonstrate filtering and normalization using a numeric field, referencing it by its `@id` as shown in the overview. Edit `numeric_field_id` and `group_field_id` as appropriate for the dataset.

In [ ]:
# Example: choose a numeric field for EDA
example_numeric_field_id = None
# Auto-select the first numeric/float/integer field from the fields metadata
for field in dataset.metadata.record_sets[0].fields:
    if field.data_type in ['Number', 'Integer', 'Float']:
        example_numeric_field_id = field.id
        break

if example_numeric_field_id is None:
    print("No numeric field found in the record set for demonstration.")
else:
    print(f"Using numeric field @id: {example_numeric_field_id}")

    # Display quick statistics
    df_num = df.copy()
    df_num[example_numeric_field_id] = pd.to_numeric(df_num[example_numeric_field_id], errors='coerce')
    print(df_num[example_numeric_field_id].describe())

    # Filter records where value > threshold (example: 10)
    threshold = 10
    filtered_df = df_num[df_num[example_numeric_field_id] > threshold]
    print(f"\nFiltered records with {example_numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the selected numeric field (z-score)
    norm_col = f"{example_numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
    print(f"\nNormalized {example_numeric_field_id} for filtered records:")
    display(filtered_df[[example_numeric_field_id, norm_col]].head())

    # Try to group by the first non-numeric field (e.g. a protein name/description)
    group_field_id = None
    for field in dataset.metadata.record_sets[0].fields:
        if field.data_type not in ['Number', 'Integer', 'Float']:
            group_field_id = field.id
            break
    if group_field_id and group_field_id in filtered_df.columns:
        # Demonstrate grouping and mean calculation
        grouped_df = filtered_df.groupby(group_field_id)[example_numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {example_numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable non-numeric group field available for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot the distribution of the chosen numeric field and a box plot grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_numeric_field_id is not None and not df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_num[example_numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {example_numeric_field_id}")
    plt.xlabel(example_numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        # Only keep top categories for display
        top_cats = df[group_field_id].value_counts().index[:8]
        box_df = df[df[group_field_id].isin(top_cats)]
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=example_numeric_field_id, data=box_df)
        plt.title(f"{example_numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization cannot be created, as no suitable numeric field was found.")

## 6. Conclusion
In this notebook, you learned how to:
- Load and explore a scientific dataset using the Croissant schema and `mlcroissant` library.
- Inspect available record sets, their fields, and reference data elements by `@id`.
- Extract tabular data and perform initial filtering and normalization using pandas.
- Visualize distributions and group-wise summaries of the data.

**You can now adapt these examples for deeper analysis and modeling on the FAIR² dataset, referencing schema entities by their `@id`s for reproducibility.**